# Repair Model Training Notebook

This notebook is the executable companion to the training README. It is organized as a short, guided workflow so you can train the repair model step by step without reading the scripts first.

What you will do here:
- train a baseline repair model on synthetic data,
- collect real ALNS states with that baseline,
- retrain with the augmented dataset,
- validate the final model path for the benchmark step.

Run the notebook from the `training` folder so the relative paths stay correct.

# Offline Repair Model Training for Hybrid ALNS

---

## 1. Purpose

This notebook explains the offline supervised-learning workflow used to train the repair model consumed by the hybrid ALNS solver.

The goal is to learn a score function $f_\theta(\phi(x))$ that ranks feasible repair actions by expected quality. The trained model is serialized as a pickle file and later loaded by the solver during inference.

The notebook is intentionally structured as a practical guide:
- train a baseline model,
- collect real ALNS trajectories,
- retrain with augmented data,
- prepare the model for benchmarking.

## 2. Data Collection Strategy

Let $x_t$ denote a partially destroyed solution state at ALNS iteration $t$, and let $\mathcal{A}(x_t)$ be the set of feasible repair actions. The collection script samples tuples $(x_t, a, y)$ where $a \in \mathcal{A}(x_t)$ and the label $y$ encodes the oracle preference for that state.

This stage is important because it reduces the gap between synthetic training data and the states encountered by the solver during real search.

For a stable dataset, keep the following parameters under control:
- `--instances`: number of sampled states or trajectories,
- `--n-min` and `--n-max`: instance size range,
- `--iterations`: number of ALNS iterations per collected run,
- `--max-negatives`: negative sampling budget,
- `--seed`: reproducibility.

## 3. Training Configuration

The main training script is `train_repair_model.py`. It uses a `GradientBoostingClassifier`, a standardized feature pipeline, weighted samples, and cross-validation to estimate generalization quality.

Recommended development settings:
- `--instances 2000`
- `--n-min 50 --n-max 200`
- `--max-negatives 3`
- `--workers 2`
- `--cv-folds 3`
- `--no-plots --no-learning-curves`

Use a larger configuration only when you want the final offline model and can afford the extra runtime.

In [ ]:
!python train_repair_model.py \
    --instances 5000 \
    --n-min 50 --n-max 200 \
    --max-negatives 5 \
    --seed 0 \
    --workers 4 \
    --output repair_model_v1.pkl

In [ ]:
!python collect_alns_states.py \
    --model-path repair_model_v1.pkl \
    --instances 500 \
    --n-min 50 --n-max 200 \
    --iterations 200 \
    --max-negatives 5 \
    --seed 1 \
    --output alns_states_v1.pkl

## 4. Collect Real ALNS States

This step uses the baseline model to generate a more realistic dataset from actual ALNS trajectories. The resulting file captures states that are close to the solver's search distribution, which helps reduce covariate shift during retraining.

The collection command below is intentionally lightweight for development. Increase `--instances` only when you need a larger augmented dataset.

In [ ]:
!python train_repair_model.py \
    --instances 2000 \
    --n-min 50 --n-max 200 \
    --max-negatives 3 \
    --seed 0 \
    --workers 2 \
    --augment-with alns_states_v1.pkl \
    --output repair_model_v2.pkl \
    --cv-folds 3 \
    --no-plots \
    --no-learning-curves

## 6. Output Contract and Validation

A successful run should produce `repair_model_v1.pkl`, `alns_states_v1.pkl`, and `repair_model_v2.pkl` in this folder unless you override the output paths.

Before moving to benchmarking, verify:
1. the baseline training step completes without exceptions,
2. the collected ALNS file is created and non-empty,
3. the retraining step prints the evaluation metrics,
4. the final model path matches the benchmark argument.

The benchmark command should be executed from the project root so the model path resolves correctly.

In [ ]:
!python ..\..\..\benchmark.py \
    --solver 5_hybrid_ml_metaheuristics/hybrid_alns/solver.py \
    --dataset falkenauer-u \
    --method-args "model_path=5_hybrid_ml_metaheuristics/hybrid_alns/models/repair_model_v2.pkl"